# Breast cancer classification

Open this notebook in Jupyter or Google Colab and run all cells from top to bottom.

Five supervised classifiers are trained on the Wisconsin Diagnostic Breast Cancer dataset from scikit-learn. Each row is one fine-needle aspirate of a breast mass. The 30 features are numeric summaries of the cell nuclei: the mean, the standard error, and the worst (largest) value of ten shape measurements such as radius, perimeter, area, smoothness, and concave points. The label is binary, malignant or benign.

Every model sees the same stratified holdout. Accuracy, precision, recall, and F1 are reported on that test set, and 5-fold accuracy on the training set is used as a second check. The stack is scikit-learn, pandas, and NumPy.


## 1. Setup

Imports stay inside that stack so the notebook runs in a clean Colab session.


In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier


## 2. Load the dataset

`load_breast_cancer` returns the Wisconsin Diagnostic table that ships with scikit-learn: 569 samples and 30 continuous features. Loading it from the library keeps the feature names, the class coding, and the row count fixed on every machine, including Colab.

In this table class `0` is malignant (212 rows) and class `1` is benign (357 rows). Benign is the majority class, about 63% of the rows, so a constant "benign" prediction would already reach about 63% accuracy. Accuracy is reported together with precision, recall, and F1 so that majority-class guessing cannot look like a successful diagnosis model.


In [2]:
dataset = load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name="diagnosis")

class_counts = y.value_counts().sort_index()
print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Malignant (0): {int(class_counts.loc[0])}")
print(f"Benign (1): {int(class_counts.loc[1])}")
print(f"Feature dtype: {X.dtypes.iloc[0]}")
print("Class names from sklearn:", ", ".join(str(name) for name in dataset.target_names))
X.head(3)


Samples: 569
Features: 30
Malignant (0): 212
Benign (1): 357
Feature dtype: float64
Class names from sklearn: malignant, benign


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


## 3. Missing values

A hospital extract of the same measurements could contain blank fields. This copy is the published Wisconsin set after that cleaning, and every column is a computed nucleus statistic, so empty cells are unlikely. The count still has to be checked, because the imputation choice depends on what is actually missing.

The audit below is on the full table, before the train/test split. Missingness is a property of the dataset. Any imputer is fit later, on the training patients only.

There are 0 missing feature values in 569 × 30 = 17,070 cells, and 0 missing labels. With nothing absent, there is no gap to fill, and the median imputer fitted in the next section leaves every training value unchanged. Median imputation is still the right method to have in place for this table: all 30 columns are numeric, and area and perimeter are right-skewed, so a mean would be pulled upward by the largest tumors. The code prints how many training cells the imputer changes. The count is zero.


In [3]:
missing_per_feature = X.isna().sum()
print(f"Missing feature values: {int(missing_per_feature.sum())} of {X.shape[0] * X.shape[1]}")
print(f"Missing labels: {int(y.isna().sum())}")
print(f"Features with at least one missing value: {int((missing_per_feature > 0).sum())}")


Missing feature values: 0 of 17070
Missing labels: 0
Features with at least one missing value: 0


## 4. One shared train/test split

Every model uses the same split: `test_size=0.2`, `random_state=42`, and `stratify=y`. The holdout is 114 patients (42 malignant, 72 benign) and the training set is 455 (170 malignant, 285 benign).

Stratification keeps the malignant share near 37% on both sides. These classes are uneven, and a plain shuffle of only 114 test rows can move that share enough to change recall for reasons that have nothing to do with the classifier. One holdout of this size can still flatter a model, which is why section 13 repeats the comparison with 5-fold accuracy on the training set.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


def split_summary(labels, name):
    counts = labels.value_counts().sort_index()
    print(
        f"{name}: n={len(labels)}  "
        f"malignant={int(counts.loc[0])}  benign={int(counts.loc[1])}  "
        f"malignant share={counts.loc[0] / len(labels):.3f}"
    )


print(f"Training matrix: {X_train.shape}")
print(f"Test matrix: {X_test.shape}")
split_summary(y_train, "Train")
split_summary(y_test, "Test")


Training matrix: (455, 30)
Test matrix: (114, 30)
Train: n=455  malignant=170  benign=285  malignant share=0.374
Test: n=114  malignant=42  benign=72  malignant share=0.368


## 5. Feature scaling

The columns do not share a unit. `worst area` runs from about 185 to about 4254, while `mean fractal dimension` sits between about 0.05 and about 0.097. The code below prints the widest and narrowest ranges. A distance, a penalty, or a coefficient that treats "one unit" as the same step on every column is then mostly a comparison of the area features.

Three of the five models are sensitive to that:

- **Logistic regression** builds a weighted sum of the features and shrinks those weights with an L2 penalty. A column with a huge raw range gets a tiny weight for the same physical change, and it dominates the penalty. Standardized columns put the coefficients in comparable units.
- **The RBF support vector machine** places the boundary using distances between patients. Two patients who differ slightly in area look far apart even when their shape features agree, unless those columns are scaled first.
- **k-nearest neighbors** is a vote of the nearest training rows under Euclidean distance. The same area columns would choose the neighbors, and smoothness or symmetry would barely move the distance.

A decision tree splits one feature at a time. Rescaling a column only moves the numeric threshold, and the patients on each side of the cut stay the same. A random forest inherits that. The trees still train on the shared scaled matrix. Every model is then scored on one design matrix, and a per-feature z-score does not remove a threshold the tree can use.

The median imputer and the standard scaler are both fit on the training patients only. The test matrix is transformed with those training statistics, so the 114 holdout patients do not shift the medians, the means, or the variances.


In [5]:
spread = X.agg(["min", "max", "std"]).T
spread["range"] = spread["max"] - spread["min"]
spread = spread.sort_values("range", ascending=False)
print("Widest features:")
print(spread.head(3).round(3).to_string())
print()
print("Narrowest features:")
print(spread.tail(3).round(6).to_string())

imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)
cells_changed = int((~np.isclose(X_train_imputed, X_train.to_numpy())).sum())
print()
print(f"Training cells changed by median imputation: {cells_changed}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)
print(
    "Training columns after scaling: "
    f"mean of means = {X_train_scaled.mean():.6f}, "
    f"mean of stds = {X_train_scaled.std(axis=0).mean():.4f}"
)


Widest features:
                min     max      std     range
worst area  185.200  4254.0  569.357  4068.800
mean area   143.500  2501.0  351.914  2357.500
area error    6.802   542.2   45.491   535.398

Narrowest features:
                              min      max       std     range
mean fractal dimension   0.049960  0.09744  0.007060  0.047480
smoothness error         0.001713  0.03113  0.003003  0.029417
fractal dimension error  0.000895  0.02984  0.002646  0.028945

Training cells changed by median imputation: 0
Training columns after scaling: mean of means = 0.000000, mean of stds = 1.0000


## 6. How the test metrics are computed

Precision, recall, and F1 treat malignant (`0`) as the positive class. On this dataset the error that matters is a malignant mass read as benign, and benign is already the majority class. Scoring the malignant label keeps precision and recall pointed at the cases a screening model has to catch.

- **Accuracy** is the fraction of test patients whose predicted label matches the true label.
- **Precision** is the fraction of patients predicted malignant who are malignant.
- **Recall** is the fraction of malignant patients who are caught.
- **F1** is the harmonic mean of that precision and recall.

The helper below stores one row per model and prints the malignant confusion counts. True malignant cases that were predicted benign are the misses.


In [6]:
MALIGNANT = 0
results = []
predictions = {}


def record(name, fitted, y_pred):
    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_malignant": precision_score(y_test, y_pred, pos_label=MALIGNANT),
        "recall_malignant": recall_score(y_test, y_pred, pos_label=MALIGNANT),
        "f1_malignant": f1_score(y_test, y_pred, pos_label=MALIGNANT),
        "train_accuracy": accuracy_score(y_train, fitted.predict(X_train_scaled)),
    }
    results.append(row)
    predictions[name] = np.asarray(y_pred)
    counts = confusion_matrix(y_test, y_pred, labels=[0, 1])
    print(
        f"{name}: accuracy {row['accuracy']:.4f}, "
        f"precision {row['precision_malignant']:.4f}, "
        f"recall {row['recall_malignant']:.4f}, "
        f"F1 {row['f1_malignant']:.4f}"
    )
    print(
        f"  Training accuracy {row['train_accuracy']:.4f}. "
        f"Malignant caught {int(counts[0, 0])} of {int(counts[0].sum())}, "
        f"missed {int(counts[0, 1])}. "
        f"Benign flagged as malignant: {int(counts[1, 0])}."
    )



## 7. Logistic regression

Logistic regression models the probability of a class as a sigmoid of a linear score, `w · x + b`. Training fits that score by minimizing L2-regularized log loss. In current scikit-learn the `lbfgs` solver does this by default (`C=1.0` is the inverse regularization strength). `max_iter=1000` gives the solver room to converge on the scaled features. `random_state=42` is set for consistency with the other estimators; the `lbfgs` fit itself is deterministic.

This is a strong match for the Wisconsin table. There are 30 numeric features and 455 training rows, so a linear score is estimable, and malignant and benign nuclei in this dataset fall close to a linear boundary. Several columns repeat the same geometry (radius, perimeter, and area move together), so the L2 penalty keeps those collinear weights from swinging. The fitted probability is also the quantity a threshold for "call it malignant" would use.


In [7]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
log_reg_pred = log_reg.predict(X_test_scaled)
record("Logistic Regression", log_reg, log_reg_pred)


Logistic Regression: accuracy 0.9825, precision 0.9762, recall 0.9762, F1 0.9762
  Training accuracy 0.9890. Malignant caught 41 of 42, missed 1. Benign flagged as malignant: 1.


## 8. Decision tree

A decision tree repeatedly cuts one feature at a threshold chosen to make the two child groups purer under Gini impurity. The prediction is the majority class in the leaf a patient reaches. `random_state=42` fixes the tie-breaking. No depth limit is set, so this comparison includes a fully grown tree.

A tree fits the way these features are used in a report. A cut on worst concave points, or on area, is a readable rule, and later cuts let the measurements interact. On 455 rows that freedom is also the risk. The tree keeps splitting until the training leaves are pure, so it can memorize this training set and then miss more of the 114 held-out patients than a model that shares strength across patients.


In [8]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train_scaled, y_train)
tree_pred = tree.predict(X_test_scaled)
record("Decision Tree", tree, tree_pred)
print(f"  Depth {tree.get_depth()}, leaves {tree.get_n_leaves()}.")


Decision Tree: accuracy 0.9123, precision 0.8478, recall 0.9286, F1 0.8864
  Training accuracy 1.0000. Malignant caught 39 of 42, missed 3. Benign flagged as malignant: 7.
  Depth 7, leaves 19.


## 9. Random forest

A random forest grows many trees, each on a bootstrap sample of the training patients and a random subset of the features at each split, then takes a majority vote. This one uses 200 trees and `random_state=42`. Two hundred trees is enough for the vote to settle on a table of this size.

The forest is the direct answer to the single tree on these data. The same axis-aligned rules are available, including interactions among the shape measurements, and the vote drops the cuts that only fit one bootstrap sample. With a few hundred rows and correlated nucleus features, that variance reduction is the property a single deep tree lacks. Per-feature scaling only moves the numeric thresholds. The forest still trains on the shared scaled matrix so every model is compared on one design.


In [9]:
forest = RandomForestClassifier(n_estimators=200, random_state=42)
forest.fit(X_train_scaled, y_train)
forest_pred = forest.predict(X_test_scaled)
record("Random Forest", forest, forest_pred)


Random Forest: accuracy 0.9561, precision 0.9512, recall 0.9286, F1 0.9398
  Training accuracy 1.0000. Malignant caught 39 of 42, missed 3. Benign flagged as malignant: 2.


## 10. Support vector machine

A support vector machine looks for the boundary with the largest margin between the classes. The RBF kernel compares patients through a smooth distance, so the boundary can curve where a flat cut is not enough. The defaults `C=1.0` and `gamma="scale"` are used. `random_state=42` is set for consistency; with these settings the RBF decision function itself is deterministic. The metrics use the predicted class.

An SVM fits this table once the features are scaled. A few hundred rows is a comfortable kernel problem, every column is numeric, and the margin is meaningful only when area and fractal dimension live on a common scale. Malignant and benign masses form two compact groups in that scaled space, which is the geometry a maximum-margin boundary is built to separate.


In [10]:
svm = SVC(kernel="rbf", random_state=42)
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)
record("SVM", svm, svm_pred)


SVM: accuracy 0.9825, precision 0.9762, recall 0.9762, F1 0.9762
  Training accuracy 0.9824. Malignant caught 41 of 42, missed 1. Benign flagged as malignant: 1.


## 11. k-nearest neighbors

k-nearest neighbors stores the training patients and, for a new row, takes the majority label among the `k=5` closest rows in Euclidean distance. Nothing is fit beyond the stored table.

It matches this problem after scaling. Two masses with similar nucleus measurements should receive the same diagnosis, and 455 rows is small enough for a direct distance computation. The 30 columns are all variants of the same ten measurements. Standardization stops area from choosing the neighbors by itself. A neighborhood of 5 is local enough to follow the boundary between malignant and benign and large enough that a single unusual neighbor does not decide the vote.


In [11]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)
record("k-NN", knn, knn_pred)


k-NN: accuracy 0.9561, precision 0.9512, recall 0.9286, F1 0.9398
  Training accuracy 0.9736. Malignant caught 39 of 42, missed 3. Benign flagged as malignant: 2.


## 12. Test-set comparison

The table uses the predictions from the shared scaled holdout. Higher malignant F1 is better. Accuracy is shown beside it because a model can post a high accuracy by handling the larger benign group well while missing malignant cases.


In [12]:
comparison = pd.DataFrame(results)
display_columns = [
    "model",
    "accuracy",
    "precision_malignant",
    "recall_malignant",
    "f1_malignant",
    "train_accuracy",
]
metric_columns = [column for column in display_columns if column != "model"]
printable = comparison[display_columns].copy()
for column in metric_columns:
    printable[column] = printable[column].map(lambda value: f"{value:.4f}")
printable = printable.rename(
    columns={
        "model": "Model",
        "accuracy": "Accuracy",
        "precision_malignant": "Precision (malignant)",
        "recall_malignant": "Recall (malignant)",
        "f1_malignant": "F1 (malignant)",
        "train_accuracy": "Train accuracy",
    }
)
print(printable.to_string(index=False))

best_f1 = comparison["f1_malignant"].max()
worst_f1 = comparison["f1_malignant"].min()
best_models = comparison.loc[comparison["f1_malignant"] == best_f1, "model"].tolist()
worst_models = comparison.loc[comparison["f1_malignant"] == worst_f1, "model"].tolist()
print()
print("Best test F1 (malignant):", ", ".join(best_models))
print("Worst test F1 (malignant):", ", ".join(worst_models))
disagree = int((predictions["Random Forest"] != predictions["k-NN"]).sum())
print(f"Test patients where random forest and k-NN disagree: {disagree}")


              Model Accuracy Precision (malignant) Recall (malignant) F1 (malignant) Train accuracy
Logistic Regression   0.9825                0.9762             0.9762         0.9762         0.9890
      Decision Tree   0.9123                0.8478             0.9286         0.8864         1.0000
      Random Forest   0.9561                0.9512             0.9286         0.9398         1.0000
                SVM   0.9825                0.9762             0.9762         0.9762         0.9824
               k-NN   0.9561                0.9512             0.9286         0.9398         0.9736

Best test F1 (malignant): Logistic Regression, SVM
Worst test F1 (malignant): Decision Tree
Test patients where random forest and k-NN disagree: 4


## 13. Five-fold check on the training set

`cross_val_score` runs 5-fold accuracy on the training patients only (`scoring="accuracy"`). The folds are stratified and shuffled with `random_state=42`, so each fold keeps the malignant share and the split can be repeated.

The imputer and the scaler sit inside the pipeline and are refit on each fold's training rows, so the row held out of a fold stays out of that fold's means and variances. The holdout table in section 12 still uses the single scaler fit on all 455 training patients. This second check asks whether the ranking survives a different cut of those same training patients.


In [13]:
estimators = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(kernel="rbf", random_state=42),
    "k-NN": KNeighborsClassifier(n_neighbors=5),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, estimator in estimators.items():
    pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", estimator),
        ]
    )
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="accuracy")
    cv_rows.append(
        {
            "model": name,
            "cv_accuracy_mean": scores.mean(),
            "cv_accuracy_std": scores.std(),
        }
    )
    fold_text = ", ".join(f"{score:.4f}" for score in scores)
    print(f"{name}: mean {scores.mean():.4f}  std {scores.std():.4f}  folds [{fold_text}]")

cv_results = pd.DataFrame(cv_rows)
print()
side_by_side = comparison[["model", "accuracy", "f1_malignant"]].merge(cv_results, on="model")
side_by_side_print = side_by_side.copy()
for column in ["accuracy", "f1_malignant", "cv_accuracy_mean", "cv_accuracy_std"]:
    side_by_side_print[column] = side_by_side_print[column].map(lambda value: f"{value:.4f}")
side_by_side_print = side_by_side_print.rename(
    columns={
        "model": "Model",
        "accuracy": "Test accuracy",
        "f1_malignant": "Test F1 (malignant)",
        "cv_accuracy_mean": "CV accuracy",
        "cv_accuracy_std": "CV std",
    }
)
print(side_by_side_print.to_string(index=False))


Logistic Regression: mean 0.9780  std 0.0098  folds [0.9670, 0.9890, 0.9780, 0.9890, 0.9670]


Decision Tree: mean 0.9165  std 0.0179  folds [0.9341, 0.9341, 0.9011, 0.8901, 0.9231]


Random Forest: mean 0.9626  std 0.0179  folds [0.9670, 0.9560, 0.9341, 0.9670, 0.9890]
SVM: mean 0.9692  std 0.0146  folds [0.9451, 0.9780, 0.9670, 0.9670, 0.9890]
k-NN: mean 0.9626  std 0.0112  folds [0.9670, 0.9670, 0.9451, 0.9560, 0.9780]

              Model Test accuracy Test F1 (malignant) CV accuracy CV std
Logistic Regression        0.9825              0.9762      0.9780 0.0098
      Decision Tree        0.9123              0.8864      0.9165 0.0179
      Random Forest        0.9561              0.9398      0.9626 0.0179
                SVM        0.9825              0.9762      0.9692 0.0146
               k-NN        0.9561              0.9398      0.9626 0.0112


## 14. Best model, worst model

On the test set, **logistic regression and the RBF SVM tie for best**. Each is correct on 112 of 114 patients: accuracy 0.9825, and malignant precision, recall, and F1 all 0.9762. Each misses one malignant case and flags one benign case. The tie fits this dataset. After scaling, both models separate two compact groups with a smooth boundary, and malignant versus benign is close to separable in these 30 nucleus measurements. The 5-fold check breaks the tie. Mean training-fold accuracy is 0.9780 for logistic regression (fold standard deviation 0.0098) and 0.9692 for the SVM (0.0146). Logistic regression is the model to keep when one choice is required: it matches the best test scores and leads the second check.

**The decision tree is worst** on the test set and in the 5-fold check. Test accuracy is 0.9123, malignant precision 0.8478, recall 0.9286, and F1 0.8864. Cross-validated accuracy is 0.9165, last among the five. The fitted tree has depth 7 and 19 leaves, and its training accuracy is 1.0, so the training patients are separated completely. On the holdout it misses 3 malignant cases and flags 7 benign cases. A single tree commits to every cut. The random forest also reaches training accuracy 1.0, because its trees are grown deep on bootstrap samples, and the vote is what the single tree lacks: the forest's test accuracy rises to 0.9561 and its malignant F1 to 0.9398.

Random forest and k-NN post the same test accuracy (0.9561) and the same malignant precision, recall, and F1 (0.9512, 0.9286, 0.9398). They disagree on 4 of the 114 test patients, so those equal scores still come from different calls that happen to add up. In the 5-fold means they tie each other at 0.9626, behind logistic regression and the SVM and ahead of the single tree. The extremes of the ranking agree across the holdout and the training folds. The middle is closer than one split of 114 patients can settle. The settings are fixed: logistic regression with `max_iter=1000`, a fully grown tree, a forest of 200 trees, an RBF SVM, and k = 5, with `random_state=42` where the estimator accepts it.
